# 1) Import Configuration and Functions:

In [0]:
%run ../common/configuration


In [0]:
%run ../common/functions

# 2) Define Pit Stops Schema:

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

pit_stops_schema = StructType([
    StructField("season", IntegerType(), False),
    StructField("round", IntegerType(), True),
    StructField("race_name", StringType(), True),
    StructField("circuit_id", StringType(), True),
    StructField("driver_id", StringType(), False),
    StructField("lap", IntegerType(), True),
    StructField("stop", IntegerType(), True),
    StructField("time", StringType(), True),
    StructField("duration", DoubleType(), True),
])

pit_stops_input_path = f"{processed_folder_path}/pit_stops/csv/pit_stops.csv"

pit_stops_df = spark.read \
    .option("header", True) \
    .schema(pit_stops_schema) \
    .csv(pit_stops_input_path)


# 3) Transform Pit Stops Data:

The steps included:

- Create Surrogate Key.
- Add Data Source and File Date.

In [0]:
from pyspark.sql.functions import lit

pit_stops_with_audit_df = pit_stops_df \
    .withColumn("data_source", lit(v_data_source)) \
    .withColumn("file_date", lit(v_file_date))

pit_stops_date_df = add_ingestion_date(pit_stops_with_audit_df)

pit_stops_final_df = add_surrogate_key(
    pit_stops_date_df,
    key_column_name="pit_stops_sk",
    hash_columns=["season", "round", "race_name", "circuit_id", "driver_id", "lap", 
                  "stop", "time", "duration"],
)

print("Final columns going into the write:", pit_stops_final_df.columns)

# 4) Save the Processed Dataset to Delta Lake:

In [0]:
pit_stops_output_path = f"{processed_folder_path}/pit_stops/delta"

spark.sql("CREATE DATABASE IF NOT EXISTS f1_processed")

upsert_if_changed_for_seasons(
    input_df=pit_stops_final_df,
    db_name="f1_processed",
    table_name="pit_stops",
    output_path=pit_stops_output_path,
    merge_key_columns=["season", "round"],
    partition_columns=["season"],
)

In [0]:
display(spark.read.format("delta").load(pit_stops_output_path))

In [0]:
build_presentation_fact(
    processed_location=f"{processed_folder_path}/pit_stops/delta",
    presentation_directory=f"{presentation_folder_path}/fact_pit_stops/delta",
    db_name="f1_presentation",
    table_name="fact_pit_stops",
    partition_columns=["season"],
)

In [0]:
display(spark.read.format("delta").load(f"{presentation_folder_path}/fact_pit_stops/delta"))

# 5) Save backup Circuits in CSV format:

In [0]:
import io
import csv

pit_stops_backup_path = f"{presentation_folder_path}/fact_pit_stops/csv/fact_pit_stops.csv"

backup_rows = [row.asDict() for row in pit_stops_final_df.collect()]
backup_fieldnames = pit_stops_final_df.columns

backup_buffer = io.StringIO()
backup_writer = csv.DictWriter(backup_buffer, fieldnames=backup_fieldnames)
backup_writer.writeheader()
backup_writer.writerows(backup_rows)

dbutils.fs.put(pit_stops_backup_path, backup_buffer.getvalue(), overwrite=True)
print(f"backup saved: {pit_stops_backup_path}")